# IE2026 Task 1b - CoT 5 - Visual Attribute Checklist - **Gemma 4 E4B**

**Requires T4 GPU.** Cross-family port of the best zero-shot Qwen notebook.

Same CoT5 prompt / parser / scoring as the Qwen2.5-VL-7B run (dev ref **CI 0.042**).
ONLY the backbone changes: `google/gemma-4-E4B-it` (effective ~4.5B, multimodal),
4-bit NF4 so it fits a single 16 GB T4.

**Question this answers:** does a *small* Gemma-4 match the levers that worked on
Qwen, or does the CoT5 gain fail to transfer (as it did on InternVL2-8B, 0.084 -> 0.084)?

| | Qwen CoT5 (ref) | This run |
|---|---|---|
| Model | Qwen2.5-VL-7B 4-bit | **Gemma-4-E4B-it 4-bit** |
| Loader | Qwen2_5_VLForConditionalGeneration | **Gemma4ForConditionalGeneration** |
| Image tokens | MAX_PIXELS=1024x28x28 | **max_soft_tokens (default 560)** |
| CoT style | attribute checklist | **identical** |
| Prompt / parser / scorer | - | **identical** |
| Dev CI ref | 0.042 | ? |


## 1. Install


In [ ]:
import os
# Gemma 4 needs a recent transformers (Gemma4ForConditionalGeneration lands in v5.13+).
os.environ['BITSANDBYTES_NOWELCOME'] = '1'
for _major in ('12', '13'):
    _src = f'/usr/local/cuda/lib64/libnvJitLink.so.{_major}'
    _dst = '/usr/local/cuda/lib64/libnvJitLink.so.13'
    if os.path.exists(_src) and not os.path.exists(_dst):
        os.symlink(_src, _dst)
        print(f'Symlinked libnvJitLink .{_major} -> .13')
        break
# Pull the latest transformers so the Gemma4 classes exist. If Kaggle's base
# image is older than the Gemma4 release, this upgrade is what makes it importable.
!pip install -q -U 'transformers>=5.13.0' accelerate bitsandbytes 2>&1 | tail -5
import transformers
print('transformers', transformers.__version__)
print('Dependencies ready.')


## 2. Config


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO_ID  = 'QCRI/AynVQA-ArabicNLP26'
TASK     = 'task1b'
LANG     = 'en'
SPLIT    = 'dev'    # 'dev' scores locally (CI printed); 'devtest' -> Codabench
RUN_ID   = 'CoT5_gemma4_e4b'

VLM_MODEL      = 'google/gemma-4-E4B-it'
QUANTIZE       = True          # 4-bit NF4 so ~8B-with-embeddings fits one T4
# Gemma 4 sizes the image to a fixed *soft token* budget instead of MAX_PIXELS.
# Allowed: 70, 140, 280(default), 560, 1120. Higher = more visual detail = slower.
# 560 ~ 1.3M px is the closest analog to Qwen's 1024x28x28 (~0.8M px) for
# fine cultural detail; drop to 280 if you hit OOM or want the fast pass.
IMG_SOFT_TOKENS = 560
MAX_NEW_TOKENS  = 384          # same as Qwen CoT5

print(f'Run: {RUN_ID}')
print(f'Split: {SPLIT} | Model: {VLM_MODEL} | soft_tokens={IMG_SOFT_TOKENS}')


In [ ]:
# HF token from Kaggle Secret or env var - never hard-code it in the notebook.
import os
from huggingface_hub import login
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('HF login: ok (token from secret/env)')
else:
    print('WARNING: no HF_TOKEN. Gemma is a GATED model - add HF_TOKEN as a '
          'Kaggle secret AND accept the licence at huggingface.co/google/gemma-4-E4B-it')


## 3. Download


In [ ]:
import json
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

jsonl = hf_hub_download(REPO_ID, filename=f'{TASK}/{SPLIT}_{LANG}.jsonl', repo_type='dataset')
records = [json.loads(l) for l in open(jsonl, encoding='utf-8') if l.strip()]
print(f'{len(records)} items | has labels: {"labels" in records[0]}')
needed = sorted({r['image'] for r in records})
paths = {}
for rel in tqdm(needed, desc='images'):
    try:
        paths[rel] = hf_hub_download(REPO_ID, filename=rel, repo_type='dataset')
    except Exception as e:
        print(f'Failed: {rel}: {e}')
print(f'Downloaded {len(paths)}/{len(needed)} images.')


## 4. Load model


In [ ]:
import torch
from transformers import Gemma4ForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

assert torch.cuda.is_available(), 'No GPU - enable T4 under Settings -> Accelerator.'
print('GPU:', torch.cuda.get_device_name(0))
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True,
) if QUANTIZE else None

# image_seq_length sets the per-image soft-token budget (Gemma's resolution knob).
processor = AutoProcessor.from_pretrained(
    VLM_MODEL, padding_side='left', image_seq_length=IMG_SOFT_TOKENS)
model = Gemma4ForConditionalGeneration.from_pretrained(
    VLM_MODEL, torch_dtype=dtype, device_map='auto',
    attn_implementation='sdpa',
    quantization_config=quant).eval()
print('Model loaded.')
for i in range(torch.cuda.device_count()):
    a = torch.cuda.memory_allocated(i)/1024**3
    t = torch.cuda.get_device_properties(i).total_memory/1024**3
    print(f'GPU {i}: {a:.1f}/{t:.1f} GB')


## 5. Prompt - CoT 5 - Visual Attribute Checklist (identical to Qwen run)


In [ ]:
PROMPT = (
    'You are a visual fact-checker examining an image from the Arab world.\n'
'Below are THREE statements. Exactly ONE is grounded in the image (True). '
'The other two are hallucinations (False).\n\n'
'Statement 1: {s0}\n'
'Statement 2: {s1}\n'
'Statement 3: {s2}\n\n'
'Instructions:\n'
'- On the VERY FIRST line write ONLY: "Answer: X" where X is 1, 2, or 3.\n'
'- For each statement evaluate these visual attributes from the image:\n'
'  (a) Colour/texture evidence for or against\n'
'  (b) Shape/form evidence for or against\n'
'  (c) Context/setting evidence for or against\n'
'- Then state which statement has the strongest combined evidence.\n'
'Do not write anything before the Answer line.'
)


## 6. Inference


In [ ]:
import re

@torch.no_grad()
def vlm_call(image_path, text, max_new_tokens=MAX_NEW_TOKENS):
    # Gemma 4 takes image + text through ONE chat-template call (no qwen_vl_utils).
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'url': image_path},
        {'type': 'text',  'text': text}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors='pt',
        add_generation_prompt=True).to(model.device)
    input_len = inputs['input_ids'].shape[-1]
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    out = processor.decode(gen[0][input_len:], skip_special_tokens=True).strip()
    del inputs, gen
    torch.cuda.empty_cache()
    return out

def parse_answer(raw):
    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    for line in lines:
        m = re.search(r'answer\s*[:\-]?\s*([123])', line, re.IGNORECASE)
        if m: return int(m.group(1))
        break
    for line in lines:
        m = re.search(r'answer\s*[:\-]?\s*([123])', line, re.IGNORECASE)
        if m: return int(m.group(1))
    for line in lines:
        if re.fullmatch(r'[123]', line): return int(line)
    return None

def predict(image_path, statements):
    text = PROMPT.format(s0=statements[0], s1=statements[1], s2=statements[2])
    raw = vlm_call(image_path, text)
    chosen = parse_answer(raw)
    if chosen is not None:
        labels = ['false','false','false']
        labels[chosen-1] = 'true'
        return labels, raw, 'joint', chosen
    # Fallback
    labels = []
    fb_raws = [raw]
    for stmt in statements:
        fb = vlm_call(image_path,
            'You are a visual fact-checker. Is this statement True or False based on the image?\n'
            f'Statement: "{stmt}"\nWrite True or False on the first line only.',
            max_new_tokens=16)
        fb_raws.append(fb)
        first = fb.strip().splitlines()[0] if fb.strip() else ''
        labels.append('true' if re.search(r'\btrue\b', first, re.IGNORECASE) else 'false')
    if labels.count('true') != 1: labels = ['true','false','false']
    return labels, ' ||| '.join(fb_raws), 'fallback', labels.index('true')+1

print('Inference functions defined.')
print('PROMPT PREVIEW:')
print('-'*60)
print(PROMPT.format(s0='<s1>', s1='<s2>', s2='<s3>'))
print('-'*60)


In [ ]:
results = []
n_joint = n_fallback = 0
for r in tqdm(records, desc=f'[{RUN_ID}]'):
    labels, raw, mode, chosen = predict(paths[r['image']], r['statements'])
    if mode == 'joint': n_joint += 1
    else: n_fallback += 1
    gold = r['labels'].index(True) if 'labels' in r else None
    results.append({
        'id': r['id'], 'country': r.get('country','?'),
        'category': r.get('category','?'),
        'gold_idx': gold, 'pred_idx': chosen-1,
        'correct': (chosen-1==gold) if gold is not None else None,
        'labels': labels, 'raw': raw, 'mode': mode,
    })
print(f'Done: {len(results)} | joint: {n_joint} | fallback: {n_fallback}')
print(f'Fallback rate: {n_fallback/len(results)*100:.1f}%')
if results[0]['correct'] is not None:
    acc = sum(r['correct'] for r in results)/len(results)
    print(f'Accuracy: {acc:.4f}  CI: {1-acc:.4f}  (Run4 dev ref: CI=0.042)')


## 7. Score


In [ ]:
scored = [r for r in results if r['correct'] is not None]
if scored:
    total = len(scored)
    q_plus = sum(r['correct'] for r in scored)
    q_minus = sum((2 if r['correct'] else 1) for r in scored)
    ci = 1 - q_plus/total
    REF = {'CI':0.042,'Comb':0.958,'Q+':0.958,'Q-':0.979}
    metrics = {'CI':ci,'Comb':q_plus/total,'Q+':q_plus/total,'Q-':q_minus/(total*2)}
    print(f'\n{RUN_ID} — {SPLIT} ({total} items)')
    print(f'{"Metric":<8} {"This run":>10}  {"Run4 dev ref":>13}  {"Delta":>8}')
    print('─'*46)
    for k in ['CI','Comb','Q+','Q-']:
        arrow = '↓' if k=='CI' else '↑'
        delta = metrics[k]-REF[k]
        better = (delta<0) if k=='CI' else (delta>0)
        flag = '✓' if better else ('=' if abs(delta)<0.001 else '✗')
        print(f'{k+" "+arrow:<8} {metrics[k]:>10.4f}  {REF[k]:>13.4f}  {delta:>+8.4f} {flag}')
    errors = total-q_plus
    print(f'\nFailures: {errors}/{total}  (Run4 dev: 21/500)')
    RUN4_FAILS = {
        '11b42e032a7f83a3','f622af57c90e5c4b','07c0c76ff3a0d676',
        '93186b379386d14f','dd3e165be23c91d6','a96c6457c0943063',
        '2478faf22a2e02ea','865bf3965e91946a','3430a7a8dcf31795',
        '60fe7c4c7479ed63','44f81fdd608f53ce','d124d94db9bfdedb',
        '77b85820e63c3278','52bf83567f90b3b9','a7859e07399b59ec',
        '9d1a4c7e01336cec','4f7c336b379d1d25','6894f784df11beb9',
        '8b6a4aa167e3d3b8','8148c168a669ba1e','640551c3f2c286f9',
    }
    fixed  = [r for r in scored if r['id'][:16] in RUN4_FAILS and r['correct']]
    broken = [r for r in scored if r['id'][:16] not in RUN4_FAILS and not r['correct']]
    print(f'Run4 failures fixed:   {len(fixed)}')
    print(f'New regressions:       {len(broken)}')
    print(f'Net:                   {len(fixed)-len(broken):+d}')
else:
    print(f"'{SPLIT}' is blind — set SPLIT='dev' to score locally.")


## 8. Save


In [ ]:
import csv, zipfile
csv_name = f'prediction_{RUN_ID}_{LANG}.csv'
with open(csv_name,'w',newline='',encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['id','statement_index','prediction'])
    for r in results:
        for si,lbl in enumerate(r['labels']):
            w.writerow([r['id'],si,lbl])
zip_name = csv_name.replace('.csv','.zip')
with zipfile.ZipFile(zip_name,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(csv_name,'prediction.csv')
print(f'Wrote {zip_name} — upload to Codabench 17051 (set SPLIT=devtest first)')
